# Microsoft Fabric PySpark Reference Notebook

## 1. Imports



In [ ]:
# Import Python utilities
import json
from datetime import datetime, timezone
from itertools import product

# Import Delta Lake support
from delta.tables import DeltaTable

# Import PySpark functions
from pyspark.sql.functions import (
    avg,
    col,
    count,
    current_timestamp,
    date_format,
    first,
    lit,
    make_date,
    max as spark_max,
    min as spark_min,
    monotonically_increasing_id,
    month,
    quarter,
    regexp_extract,
    regexp_replace,
    round as spark_round,
    row_number,
    substring,
    sum as spark_sum,
    to_date,
    to_timestamp,
    trim,
    upper,
    when,
    year
)

# Import PySpark data types
from pyspark.sql.types import (
    DoubleType,
    StringType,
    StructField,
    StructType
)

# Import window functions
from pyspark.sql.window import Window



## 2. Pipeline Parameters and Partitioned Paths



In [ ]:
# Use today's UTC date when a pipeline value is not supplied
load_date = ""

if not load_date:
    load_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")

# Build a date-partitioned Bronze path
bronze_path = (
    f"Files/Bronze/SourceName/Raw/"
    f"load_date={load_date}/source_file.csv"
)

print(f"Load date: {load_date}")
print(f"Bronze path: {bronze_path}")



## 3. Source Reading Patterns



In [ ]:
# Read a CSV file from a date-partitioned Bronze folder
df_csv = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(
            f"Files/Bronze/PPR/Extracted/"
            f"load_date={load_date}/PPR-ALL.csv"
        )
)

display(df_csv.limit(10))



In [ ]:
# Read a whole JSON-stat file as one text value
json_path = (
    f"Files/Bronze/CSO/VAC14/Raw/"
    f"load_date={load_date}/VAC14.json"
)

raw_text = (
    spark.read
        .option("wholetext", "true")
        .text(json_path)
        .first()["value"]
)

payload = json.loads(raw_text)
dataset = payload[0] if isinstance(payload, list) else payload

print(f"Dataset: {dataset.get('label', 'Unknown')}")
print(f"Dimensions: {dataset.get('size')}")



In [ ]:
# Read several normalised Parquet extracts without repeating the load code
planning_table_names = [
    "PlanningApplication",
    "PlanningAuthority",
    "ApplicationType",
    "ApplicationStatus",
    "DecisionType",
    "LandUseType"
]

planning_frames = {
    table_name: spark.read.format("parquet").load(
        f"Files/Bronze/PlanningSQL/{table_name}/"
        f"load_date={load_date}/{table_name}.parquet"
    )
    for table_name in planning_table_names
}

df_planning_application = planning_frames["PlanningApplication"]
df_planning_authority = planning_frames["PlanningAuthority"]
df_application_type = planning_frames["ApplicationType"]
df_application_status = planning_frames["ApplicationStatus"]
df_decision_type = planning_frames["DecisionType"]
df_land_use_type = planning_frames["LandUseType"]

for table_name, dataframe in planning_frames.items():
    print(f"{table_name}: {dataframe.count():,}")



In [ ]:
# Read a Delta path and a registered Lakehouse table
df_delta = spark.read.format("delta").load(
    "Files/Silver/planning_applications"
)

df_table = spark.table("stg_dim_date")



## 4. Cleaning and Standardisation



In [ ]:
# Clean raw PPR columns and create a standardised Silver dataset
df_ppr_silver = (
    df_csv
    .withColumn(
        "SalePrice",
        regexp_replace(
            col("Price (€)"),
            r"[€,]",
            ""
        ).cast("decimal(18,2)")
    )
    .withColumn(
        "SaleDate",
        to_date(
            col("Date of Sale (dd/mm/yyyy)"),
            "dd/MM/yyyy"
        )
    )
    .select(
        "SaleDate",
        "Address",
        "County",
        "Eircode",
        "SalePrice",
        col("Not Full Market Price").alias("IsFullMarketPrice"),
        col("VAT Exclusive").alias("IsVATExclusive"),
        col("Description of Property").alias("PropertyType"),
        col("Property Size Description").alias("PropertySizeCategory")
    )
    .withColumn(
        "IsFullMarketPrice",
        when(col("IsFullMarketPrice") == "No", lit("Yes"))
        .otherwise(lit("No"))
    )
    .withColumn(
        "PropertyType",
        when(
            col("PropertyType").contains("Ath"),
            lit("Second-Hand Dwelling house / Apartment")
        )
        .when(
            col("PropertyType").contains("Nua"),
            lit("New Dwelling house / Apartment")
        )
        .otherwise(col("PropertyType"))
    )
    .dropDuplicates()
    .withColumn("LoadTimestamp", current_timestamp())
    .withColumn("SourceSystem", lit("PPR"))
)

display(df_ppr_silver.limit(20))



In [ ]:
# Validate mandatory values and basic business rules
validation_results = {
    "Row count": df_ppr_silver.count(),
    "Missing sale dates": df_ppr_silver.filter(
        col("SaleDate").isNull()
    ).count(),
    "Missing sale prices": df_ppr_silver.filter(
        col("SalePrice").isNull()
    ).count(),
    "Non-positive prices": df_ppr_silver.filter(
        col("SalePrice") <= 0
    ).count()
}

for check_name, result in validation_results.items():
    print(f"{check_name}: {result:,}")



## 5. Flattening JSON-stat Data



In [ ]:
# Extract JSON-stat dimensions and values
dimensions = dataset["dimension"]

statistic_dimension = dimensions["STATISTIC"]
quarter_dimension = dimensions["TLIST(Q1)"]
local_authority_dimension = dimensions["C03789V04537"]

statistic_codes = statistic_dimension["category"]["index"]
quarter_codes = quarter_dimension["category"]["index"]
local_authority_codes = local_authority_dimension["category"]["index"]

statistic_labels = statistic_dimension["category"]["label"]
quarter_labels = quarter_dimension["category"]["label"]
local_authority_labels = local_authority_dimension["category"]["label"]
statistic_units = statistic_dimension["category"]["unit"]

values = dataset["value"]
source_updated = dataset.get("updated")

expected_value_count = (
    len(statistic_codes)
    * len(quarter_codes)
    * len(local_authority_codes)
)

if len(values) != expected_value_count:
    raise ValueError(
        "JSON-stat dimensions do not match the number of values."
    )

# Flatten the positional values into individual records
records = []

for value_position, (
    statistic_code,
    quarter_code,
    local_authority_code
) in enumerate(
    product(
        statistic_codes,
        quarter_codes,
        local_authority_codes
    )
):
    value = values[value_position]
    unit = statistic_units.get(statistic_code, {}).get("label")

    records.append(
        (
            statistic_code,
            statistic_labels[statistic_code],
            quarter_code,
            quarter_labels[quarter_code],
            local_authority_code,
            local_authority_labels[local_authority_code],
            unit,
            float(value) if value is not None else None,
            source_updated,
            load_date
        )
    )

print(f"Records created: {len(records):,}")

# Define an explicit schema for the flattened records
json_stat_schema = StructType(
    [
        StructField("StatisticCode", StringType(), False),
        StructField("Statistic", StringType(), False),
        StructField("QuarterCode", StringType(), False),
        StructField("Quarter", StringType(), False),
        StructField("LocalAuthorityCode", StringType(), False),
        StructField("LocalAuthority", StringType(), False),
        StructField("Unit", StringType(), True),
        StructField("Value", DoubleType(), True),
        StructField("SourceUpdatedTimestamp", StringType(), True),
        StructField("LoadDate", StringType(), False)
    ]
)

df_json_stat_silver = (
    spark.createDataFrame(records, schema=json_stat_schema)
    .withColumn(
        "SourceUpdatedTimestamp",
        to_timestamp(
            col("SourceUpdatedTimestamp"),
            "yyyy-MM-dd'T'HH:mm:ss.SSSX"
        )
    )
    .withColumn("LoadDate", to_date(col("LoadDate")))
    .withColumn("IngestedTimestamp", current_timestamp())
)

display(df_json_stat_silver.limit(20))



## 6. Denormalising Relational Source Tables



In [ ]:
# Join normalised source tables into one analytical dataset
df_planning_joined = (
    df_planning_application
    .join(
        df_planning_authority,
        on="PlanningAuthorityID",
        how="left"
    )
    .join(
        df_application_type,
        on="ApplicationTypeID",
        how="left"
    )
    .join(
        df_application_status,
        on="ApplicationStatusID",
        how="left"
    )
    .join(
        df_decision_type,
        on="DecisionTypeID",
        how="left"
    )
    .join(
        df_land_use_type,
        on="LandUseTypeID",
        how="left"
    )
)

print(f"Source rows: {df_planning_application.count():,}")
print(f"Joined rows: {df_planning_joined.count():,}")



In [ ]:
# Separate acceptable source nulls from broken lookup matches
reference_checks = [
    ("Application status", "ApplicationStatusID", "ApplicationStatusName"),
    ("Decision type", "DecisionTypeID", "DecisionTypeName"),
    ("Land-use type", "LandUseTypeID", "LandUseTypeName")
]

for label, id_column, name_column in reference_checks:
    null_source_ids = df_planning_joined.filter(
        col(id_column).isNull()
    ).count()

    unmatched_ids = df_planning_joined.filter(
        col(id_column).isNotNull()
        & col(name_column).isNull()
    ).count()

    print(f"{label} - null source IDs: {null_source_ids:,}")
    print(f"{label} - unmatched IDs: {unmatched_ids:,}")



In [ ]:
# Validate identifiers, duplicates and geographic coordinates
missing_source_ids = df_planning_joined.filter(
    col("SourceObjectID").isNull()
).count()

duplicate_source_ids = (
    df_planning_joined
    .groupBy("SourceObjectID")
    .count()
    .filter(col("count") > 1)
    .count()
)

invalid_coordinates = df_planning_joined.filter(
    col("Latitude").isNull()
    | col("Longitude").isNull()
    | ~col("Latitude").between(51.0, 56.0)
    | ~col("Longitude").between(-11.0, -5.0)
).count()

print(f"Missing source IDs: {missing_source_ids:,}")
print(f"Duplicate source IDs: {duplicate_source_ids:,}")
print(f"Missing or invalid coordinates: {invalid_coordinates:,}")



## 7. Gold Business Rules and Standardised Categories



In [ ]:
# Create reusable analytical fields and decision categories
decision_text = upper(col("DecisionTypeName"))

df_planning_gold = (
    df_planning_joined
    .withColumn("ApplicationYear", year(col("ReceivedDate")))
    .withColumn("ApplicationMonth", month(col("ReceivedDate")))
    .withColumn(
        "ApplicationYearMonth",
        date_format(col("ReceivedDate"), "yyyy-MM")
    )
    .withColumn(
        "OneOffHouseCategory",
        when(col("OneOffHouse") == True, lit("Yes"))
        .when(col("OneOffHouse") == False, lit("No"))
        .otherwise(lit("Not recorded"))
    )
    .withColumn(
        "DecisionOutcome",
        when(
            col("DecisionTypeName").isNull()
            | col("DecisionTypeName").isin("Not recorded", "N/A"),
            lit("No decision recorded")
        )
        .when(decision_text.contains("INVA"), lit("Invalid"))
        .when(decision_text.contains("WITHDRAW"), lit("Withdrawn"))
        .when(decision_text.contains("SPLIT"), lit("Split decision"))
        .when(decision_text.contains("REFUS"), lit("Refused"))
        .when(
            decision_text.contains("GRANT")
            | decision_text.contains("CONDITIONAL")
            | decision_text.contains("APPROV"),
            lit("Granted")
        )
        .when(decision_text.contains("EXEMPT"), lit("Exempt"))
        .when(
            decision_text.contains("ADDITIONAL INFORMATION")
            | decision_text.contains("FURTHER INFORMATION")
            | decision_text.contains("REQUEST AI")
            | decision_text.contains("REQ AI"),
            lit("Further information")
        )
        .otherwise(lit("Other"))
    )
    .withColumn(
        "DecisionRecorded",
        when(
            col("DecisionOutcome") == "No decision recorded",
            lit(0)
        ).otherwise(lit(1))
    )
    .withColumn("GoldLoadTimestamp", current_timestamp())
)

display(
    df_planning_gold
    .groupBy("DecisionOutcome")
    .agg(count("*").alias("ApplicationCount"))
    .orderBy(col("ApplicationCount").desc())
)



## 8. Pivoting and Aggregating Measures



In [ ]:
# Pivot statistic rows into separate business measures
df_local_authority = (
    df_json_stat_silver
    .groupBy(
        "QuarterCode",
        "Quarter",
        "LocalAuthorityCode",
        "LocalAuthority",
        "SourceUpdatedTimestamp",
        "LoadDate"
    )
    .pivot(
        "StatisticCode",
        ["VAC14C01", "VAC14C02", "VAC14C03"]
    )
    .agg(first("Value"))
    .withColumnRenamed("VAC14C01", "VacantDwellings")
    .withColumnRenamed("VAC14C02", "DwellingStock")
    .withColumnRenamed("VAC14C03", "ReportedVacancyRate")
)

# Map local authorities to counties
df_county_mapped = df_local_authority.withColumn(
    "County",
    when(
        col("LocalAuthority").isin(
            "Dublin City Council",
            "Dún Laoghaire Rathdown County Council",
            "Fingal County Council",
            "South Dublin County Council"
        ),
        lit("Dublin")
    )
    .when(
        col("LocalAuthority").isin(
            "Cork City Council",
            "Cork County Council"
        ),
        lit("Cork")
    )
    .when(
        col("LocalAuthority").isin(
            "Galway City Council",
            "Galway County Council"
        ),
        lit("Galway")
    )
    .when(
        col("LocalAuthority") == "Limerick City & County Council",
        lit("Limerick")
    )
    .when(
        col("LocalAuthority") == "Waterford City & County Council",
        lit("Waterford")
    )
    .otherwise(
        regexp_replace(
            col("LocalAuthority"),
            " County Council$",
            ""
        )
    )
)

# Aggregate local-authority values to county level
df_vacancy_gold = (
    df_county_mapped
    .groupBy("QuarterCode", "Quarter", "County")
    .agg(
        spark_sum("VacantDwellings").alias("VacantDwellings"),
        spark_sum("DwellingStock").alias("DwellingStock"),
        spark_max("SourceUpdatedTimestamp").alias(
            "SourceUpdatedTimestamp"
        ),
        spark_max("LoadDate").alias("LoadDate")
    )
    .withColumn(
        "VacancyRate",
        spark_round(
            col("VacantDwellings") * lit(100.0)
            / col("DwellingStock"),
            1
        )
    )
    .withColumn(
        "Year",
        regexp_extract(col("Quarter"), r"^(\d{4})", 1).cast("int")
    )
    .withColumn(
        "QuarterNumber",
        regexp_extract(col("Quarter"), r"Q([1-4])$", 1).cast("int")
    )
    .withColumn(
        "QuarterStartDate",
        make_date(
            col("Year"),
            ((col("QuarterNumber") - 1) * 3 + 1).cast("int"),
            lit(1)
        )
    )
    .withColumn("GoldLoadTimestamp", current_timestamp())
)

display(df_vacancy_gold.limit(20))



## 9. Delta Lake Writes and Incremental Merge



In [ ]:
# Replace a complete snapshot and enforce its latest schema
df_ppr_silver.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .save("Files/Silver/ppr_property_sales")



In [ ]:
# Merge new and changed records into an existing Delta table
target_path = "Files/Gold/gold_planning_applications"

if DeltaTable.isDeltaTable(spark, target_path):
    target_delta = DeltaTable.forPath(spark, target_path)

    target_delta.alias("target") \
        .merge(
            df_planning_gold.alias("source"),
            "target.SourceObjectID = source.SourceObjectID"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()

    print("Delta merge completed.")

else:
    df_planning_gold.write \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .format("delta") \
        .save(target_path)

    print("Initial Delta table created.")



## 10. Dimensional Modelling



In [ ]:
# Build a dimension with a generated surrogate key
window_spec = Window.orderBy("County")

dim_county = (
    df_ppr_silver
    .select("County")
    .filter(col("County").isNotNull())
    .distinct()
    .withColumn("CountyKey", row_number().over(window_spec))
    .select(
        "CountyKey",
        col("County").alias("CountyName")
    )
)

# Add an unknown member for missing source references
unknown_county = spark.createDataFrame(
    [(0, "Not recorded")],
    "CountyKey INT, CountyName STRING"
)

dim_county = (
    unknown_county
    .unionByName(dim_county)
    .orderBy("CountyKey")
)

display(dim_county)



In [ ]:
# Build a complete shared date dimension
calendar_dates = spark.sql(
    """
    SELECT explode(
        sequence(
            to_date('2010-01-01'),
            to_date('2030-12-31'),
            interval 1 day
        )
    ) AS FullDate
    """
)

dim_date = (
    calendar_dates
    .select(
        date_format(
            col("FullDate"),
            "yyyyMMdd"
        ).cast("int").alias("DateKey"),
        col("FullDate"),
        year(col("FullDate")).cast("int").alias("Year"),
        quarter(col("FullDate")).cast("int").alias("Quarter"),
        month(col("FullDate")).cast("int").alias("Month"),
        date_format(col("FullDate"), "MMMM").alias("MonthName")
    )
    .orderBy("FullDate")
)

print(f"Date-dimension rows: {dim_date.count():,}")



In [ ]:
# Build a fact table by replacing business values with dimension keys
# Use a stable source key or hash instead for an incrementally maintained fact
fact_property_sales = (
    df_ppr_silver.alias("sales")
    .join(
        dim_date.alias("date"),
        col("sales.SaleDate") == col("date.FullDate"),
        "left"
    )
    .join(
        dim_county.alias("county"),
        col("sales.County") == col("county.CountyName"),
        "left"
    )
    .select(
        monotonically_increasing_id().alias("PropertySaleKey"),
        col("date.DateKey"),
        col("county.CountyKey"),
        col("sales.Address"),
        col("sales.Eircode"),
        col("sales.SalePrice"),
        col("sales.IsFullMarketPrice"),
        col("sales.LoadTimestamp")
    )
)

display(fact_property_sales.limit(20))



In [ ]:
# Find fact keys that do not exist in the related dimension
missing_county_keys = (
    fact_property_sales
    .select("CountyKey")
    .distinct()
    .join(
        dim_county.select("CountyKey"),
        on="CountyKey",
        how="left_anti"
    )
    .count()
)

print(f"Missing county keys: {missing_county_keys}")



## 11. Lakehouse Staging Tables



In [ ]:
# Write several staging tables without repeating the write block
staging_tables = {
    "stg_dim_date": dim_date,
    "stg_dim_county": dim_county,
    "stg_fact_property_sales": fact_property_sales
}

for table_name, dataframe in staging_tables.items():
    dataframe.write \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .format("delta") \
        .saveAsTable(table_name)

    print(f"Saved {table_name}: {dataframe.count():,} rows")



## 12. Reusable Data-Quality Profile



In [ ]:
# Profile row counts, duplicates and mandatory-column nulls
def profile_dataframe(
    dataframe,
    key_columns,
    mandatory_columns
):
    row_count = dataframe.count()

    duplicate_count = (
        dataframe
        .groupBy(*key_columns)
        .count()
        .filter(col("count") > 1)
        .count()
    )

    print(f"Rows: {row_count:,}")
    print(f"Duplicate keys: {duplicate_count:,}")

    for column_name in mandatory_columns:
        null_count = dataframe.filter(
            col(column_name).isNull()
        ).count()

        print(f"Missing {column_name}: {null_count:,}")


# Apply the reusable profile
profile_dataframe(
    dataframe=df_ppr_silver,
    key_columns=["SaleDate", "Address", "SalePrice"],
    mandatory_columns=["SaleDate", "County", "SalePrice"]
)

